# 05_silver_patient_match
Links the patients and visits in the legacy HL7 feed to their FHIR records. The two sources use different identifiers:
HL7 carries a hospital MRN (and *sometimes* the enterprise ID); FHIR uses its own patient ID.

| Output table | What it holds |
| --- | --- |
| `health.silver.hl7_patients` | one row per HL7 patient (MRN) with matching attributes |
| `health.silver.patient_match_candidates` | every HL7 patient scored against every FHIR patient |
| `health.silver.patient_xref` | the accepted links: MRN → FHIR patient ID, with method and score |
| `health.silver.patient_match_review` | HL7 patients with no confident match, for a human to review |
| `health.silver.encounter_xref` | HL7 visit → FHIR encounter |

**Two passes**, as a real master patient index would do:
1. **Deterministic:** if any message for the patient carried the enterprise ID, link on it directly.
2. **Probabilistic (demographic):** otherwise, score every FHIR patient on name, date of birth, sex and ZIP, and accept the best only if it is high *and* clearly ahead of the runner-up.

The results are then checked against `data/validation/hl7_ground_truth.csv`, which the matching never reads.

## Step 1: one row per HL7 patient

In [0]:
%sql
CREATE OR REPLACE TABLE health.silver.hl7_patients
COMMENT 'One row per legacy MRN from the HL7 feed, with the attributes used for matching.'
AS
SELECT
  mrn,
  max(enterprise_id)               AS enterprise_id,   -- found on ANY of the patient's visits
  max_by(family_norm, admit_ts)    AS family_norm,
  max_by(given_norm,  admit_ts)    AS given_norm,
  max_by(birth_date,  admit_ts)    AS birth_date,
  max_by(sex,         admit_ts)    AS sex,
  max_by(zip,         admit_ts)    AS zip,
  count(*)                         AS visit_count
FROM health.silver.hl7_encounters
GROUP BY mrn;

SELECT COUNT(*) AS hl7_patients,                                          -- expect 88
       COUNT(enterprise_id) AS with_enterprise_id,                        -- expect 86
       COUNT(*) - COUNT(enterprise_id) AS need_demographic_matching       -- expect 2
FROM health.silver.hl7_patients;

Note what happened: **317 visits** had no enterprise ID, but at the *patient* level only **2 patients** never had it on any message.
Pooling identifiers across all of a patient's messages is the first, cheapest step in record linkage.

## Step 2: score every HL7 patient against every FHIR patient

| Attribute | Exact match | Close match (≤ 2 typos) |
| --- | --- | --- |
| Family name | 30 | 20 |
| Given name | 25 | 15 |
| Date of birth | 30 | 0 |
| ZIP | 10 | 0 |
| Sex | 5 | 0 |

Scores for *all* 88 patients are kept, so the demographic method can also be checked against the enterprise-ID links.

In [0]:
%sql
CREATE OR REPLACE TABLE health.silver.patient_match_candidates
COMMENT 'Every HL7 patient scored against every FHIR patient on name, date of birth, sex and ZIP.'
AS
WITH scored AS (
  SELECT
    h.mrn,
    f.patient_id AS fhir_patient_id,
      CASE WHEN h.family_norm = f.family_norm THEN 30
           WHEN levenshtein(h.family_norm, f.family_norm) <= 2 THEN 20 ELSE 0 END
    + CASE WHEN h.given_norm = f.given_norm THEN 25
           WHEN levenshtein(h.given_norm, f.given_norm) <= 2 THEN 15 ELSE 0 END
    + CASE WHEN h.birth_date = f.birth_date THEN 30 ELSE 0 END
    + CASE WHEN h.zip = f.zip THEN 10 ELSE 0 END
    + CASE WHEN h.sex = f.sex THEN 5 ELSE 0 END AS match_score
  FROM health.silver.hl7_patients h
  CROSS JOIN health.silver.fhir_patients f
)
SELECT
  *,
  row_number() OVER (PARTITION BY mrn ORDER BY match_score DESC, fhir_patient_id) AS candidate_rank,
  lead(match_score) OVER (PARTITION BY mrn ORDER BY match_score DESC, fhir_patient_id) AS runner_up_score
FROM scored;

## Step 3: accept the links
A demographic match is accepted only if the score is **at least 85** and the runner-up is **at least 10 points behind**. Anything else goes to the review table instead of being guessed.

In [0]:
%sql
CREATE OR REPLACE TABLE health.silver.patient_xref
COMMENT 'Accepted links between legacy MRNs and FHIR patient IDs, with the method and score.'
AS
WITH pass1 AS (
  SELECT h.mrn, f.patient_id AS fhir_patient_id, 'enterprise_id' AS match_method, 100 AS match_score
  FROM health.silver.hl7_patients h
  JOIN health.silver.fhir_patients f ON h.enterprise_id = f.patient_id
),
pass2 AS (
  SELECT c.mrn, c.fhir_patient_id, 'demographic' AS match_method, c.match_score
  FROM health.silver.patient_match_candidates c
  WHERE c.candidate_rank = 1
    AND c.match_score >= 85
    AND coalesce(c.runner_up_score, 0) <= c.match_score - 10
    AND c.mrn NOT IN (SELECT mrn FROM pass1)
)
SELECT *, current_timestamp() AS matched_at
FROM (SELECT * FROM pass1 UNION ALL SELECT * FROM pass2);

CREATE OR REPLACE TABLE health.silver.patient_match_review
COMMENT 'HL7 patients with no confident match: best candidate shown for manual review.'
AS
SELECT c.*
FROM health.silver.patient_match_candidates c
WHERE c.candidate_rank = 1
  AND c.mrn NOT IN (SELECT mrn FROM health.silver.patient_xref);

SELECT match_method, COUNT(*) AS patients, MIN(match_score) AS lowest_score
FROM health.silver.patient_xref
GROUP BY match_method;   -- expect enterprise_id 86, demographic 2 (both scoring 100)

## Step 4: link the visits, one to one
A visit matches a FHIR encounter for the same patient and facility starting in the same minute (HL7 timestamps are sometimes truncated to the minute).

That alone is not enough: a patient can have **two encounters at the same facility starting in the same minute**, for example an emergency visit that becomes an inpatient admission. Joining on start time would link both visits to the same encounter and duplicate fact rows. So the match also uses the **encounter class** and **end time**, and `row_number()` on both sides guarantees each encounter is used once.
Pass A requires everything to agree; Pass B links the leftovers (mostly visits whose discharge time was missing) on start time and class.

In [0]:
%sql
CREATE OR REPLACE TABLE health.silver.encounter_xref
COMMENT 'One-to-one links between HL7 visits and FHIR encounters, with the rule that made each link.'
AS
WITH h AS (
  SELECT
    e.facility_code, e.visit_number, x.fhir_patient_id,
    upper(e.facility_name)                    AS fac,
    date_trunc('MINUTE', e.admit_ts)          AS start_min,
    CASE e.patient_class WHEN 'I' THEN 'IMP' WHEN 'E' THEN 'EMER' ELSE 'AMB' END AS cls,
    date_trunc('MINUTE', e.discharge_ts)      AS end_min
  FROM health.silver.hl7_encounters e
  JOIN health.silver.patient_xref x ON e.mrn = x.mrn
),
f AS (
  SELECT
    encounter_id, patient_id AS fhir_patient_id,
    upper(facility_name)                      AS fac,
    date_trunc('MINUTE', start_ts)            AS start_min,
    class_code                                AS cls,
    date_trunc('MINUTE', end_ts)              AS end_min
  FROM health.silver.fhir_encounters
),
-- Pass A: patient, facility, start minute, class AND end minute all agree
h1 AS (SELECT *, row_number() OVER (PARTITION BY fhir_patient_id, fac, start_min, cls, end_min ORDER BY visit_number) AS rn
       FROM h WHERE end_min IS NOT NULL),
f1 AS (SELECT *, row_number() OVER (PARTITION BY fhir_patient_id, fac, start_min, cls, end_min ORDER BY encounter_id) AS rn
       FROM f),
pass_a AS (
  SELECT facility_code, visit_number, encounter_id, fhir_patient_id, 'start + class + end' AS match_rule
  FROM h1 JOIN f1 USING (fhir_patient_id, fac, start_min, cls, end_min, rn)
),
-- Pass B: whatever is left (e.g. the discharge time was missing) on start minute and class
h2 AS (SELECT *, row_number() OVER (PARTITION BY fhir_patient_id, fac, start_min, cls ORDER BY visit_number) AS rn
       FROM h LEFT ANTI JOIN pass_a USING (facility_code, visit_number)),
f2 AS (SELECT *, row_number() OVER (PARTITION BY fhir_patient_id, fac, start_min, cls ORDER BY encounter_id) AS rn
       FROM f LEFT ANTI JOIN pass_a USING (encounter_id)),
pass_b AS (
  SELECT facility_code, visit_number, encounter_id, fhir_patient_id, 'start + class' AS match_rule
  FROM h2 JOIN f2 USING (fhir_patient_id, fac, start_min, cls, rn)
)
SELECT * FROM pass_a
UNION ALL
SELECT * FROM pass_b;

-- Expect 1,799 links, and every encounter used at most once
SELECT match_rule, COUNT(*) AS links, COUNT(DISTINCT encounter_id) AS distinct_encounters
FROM health.silver.encounter_xref
GROUP BY match_rule;

## Step 5: check the results against the answer key
The ground-truth file lives in the repo at `data/validation/`. It is loaded only here, after matching is finished.

In [0]:
import os
import pandas as pd

gt_path = os.path.abspath("../data/validation/hl7_ground_truth.csv")
truth = pd.read_csv(gt_path, dtype=str)          # dtype=str keeps the MRNs' leading zeros
spark.createDataFrame(truth).createOrReplaceTempView("ground_truth")
print(f"Loaded {len(truth):,} ground-truth rows from {gt_path}")

In [0]:
%sql
-- Patient linkage accuracy. Expect 88 linked, 88 correct, 0 wrong, 0 unmatched.
WITH truth AS (SELECT DISTINCT legacy_mrn AS mrn, synthea_patient_id FROM ground_truth)
SELECT
  COUNT(*)                                                                AS hl7_patients,
  COUNT(x.mrn)                                                            AS linked,
  SUM(CASE WHEN x.fhir_patient_id =  t.synthea_patient_id THEN 1 ELSE 0 END) AS correct,
  SUM(CASE WHEN x.fhir_patient_id <> t.synthea_patient_id THEN 1 ELSE 0 END) AS wrong,
  COUNT(*) - COUNT(x.mrn)                                                 AS unmatched
FROM truth t
LEFT JOIN health.silver.patient_xref x USING (mrn)

In [0]:
%sql
-- Visit linkage accuracy. Expect 1,799 linked and 1,795 correct.
-- The 4 'wrong' links are 2 pairs of encounters with identical patient, facility, class, start AND end minute:
-- nothing in either source can tell them apart, so any one-to-one pairing is as good as another.
SELECT
  COUNT(*)                                                               AS hl7_visits,
  COUNT(e.encounter_id)                                                  AS linked,
  SUM(CASE WHEN e.encounter_id = g.fhir_encounter_id THEN 1 ELSE 0 END)  AS correct
FROM ground_truth g
LEFT JOIN health.silver.encounter_xref e ON e.visit_number = g.visit_number

In [0]:
%sql
-- Independent check: would demographics ALONE have found the same patients as the enterprise ID?
SELECT
  COUNT(*)                                                                 AS enterprise_id_links,
  SUM(CASE WHEN c.fhir_patient_id = x.fhir_patient_id THEN 1 ELSE 0 END)  AS demographic_agrees
FROM health.silver.patient_xref x
JOIN health.silver.patient_match_candidates c
  ON c.mrn = x.mrn AND c.candidate_rank = 1
WHERE x.match_method = 'enterprise_id'   -- expect 86 and 86